# Load the datasets

In [1]:
from datasets import load_dataset

configs = [
    "learned_hands_benefits",
    "learned_hands_business",
    "learned_hands_consumer",
    "learned_hands_courts",
    "learned_hands_crime",
    "learned_hands_divorce",
    "learned_hands_domestic_violence",
    "learned_hands_education",
    "learned_hands_employment",
    "learned_hands_estates",
    "learned_hands_family",
    "learned_hands_health",
    "learned_hands_housing",
    "learned_hands_immigration",
    "learned_hands_torts",
    "learned_hands_traffic",
]

datasets = {
    name: load_dataset("nguha/legalbench", name)
    for name in configs
}

c:\Users\pc\Desktop\Stuff\Programming\nlp\projects\Legal Scope Guardrail\legal-scope-guardrail\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Merge train and test splits for each dataset

In [2]:
from datasets import concatenate_datasets

merged_datasets = {
    name: concatenate_datasets([ds["train"], ds["test"]])
    for name, ds in datasets.items()
}

# Remove negative examples

In [3]:
positive_datasets = {
    name: ds.filter(lambda x: x["answer"].lower() == "yes")
    for name, ds in merged_datasets.items()
}

# Merge into a unified dataset

In [4]:
def to_domain(example, domain):
    return {
        "text": example["text"],
        "domain": domain
    }

all_data = []

for name, ds in positive_datasets.items():
    ds = ds.map(lambda x: to_domain(x, name))
    ds = ds.remove_columns([col for col in ds.column_names if col not in ["text", "domain"]])
    all_data.append(ds)

from datasets import concatenate_datasets
final_ds = concatenate_datasets(all_data)

# Get hard negatives

In [5]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)  

True
12.1


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

all_texts = final_ds["text"]
all_domains = np.array(final_ds["domain"])
all_embeddings = model.encode(all_texts, convert_to_numpy=True, normalize_embeddings=True)

sim_matrix = all_embeddings @ all_embeddings.T

def get_hard_negative(anchor_idx, domain):
    sims = sim_matrix[anchor_idx].copy()

    invalid_mask = (
        (all_domains == domain) |
        (np.arange(len(all_texts)) == anchor_idx) |
        (np.array(all_texts) == all_texts[anchor_idx])
    )

    sims[invalid_mask] = -np.inf

    best_idx = np.argmax(sims)

    if sims[best_idx] == -np.inf:
        return None

    return all_texts[best_idx]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6690.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from random import choice

domain_to_texts = {}
for d in configs:
    domain_to_texts[d] = final_ds.filter(lambda x: x["domain"] == d)["text"]


def make_triplet(example, idx):
    anchor = example["text"]
    domain = example["domain"]

    texts = domain_to_texts[domain]

    if len(texts) > 1:
        positive = choice(texts)
        while positive == anchor:
            positive = choice(texts)
    else:
        positive = anchor

    negative = get_hard_negative(idx, domain)

    return {
        "anchor": anchor,
        "positive": positive,
        "negative": negative
    }


triplet_ds = final_ds.map(make_triplet, with_indices=True)

Map: 100%|██████████| 5603/5603 [10:59<00:00,  8.49 examples/s]


In [8]:
triplet_ds.save_to_disk("triplet_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 5603/5603 [00:00<00:00, 278853.83 examples/s]


In [9]:
from datasets import load_from_disk

loaded = load_from_disk("triplet_dataset")

In [10]:
loaded[0]

{'text': '(suicide tw throughout this post) Background: I have struggled with depression/bipolar for a number of years, and just over a year ago it caused me to be unemployed for 3 months as I was doing outpatient and recovering.  Found a new job that I love in many ways and really care about, but it didn\'t stop the number of suicide attempts.  I had to take multiple leaves of absences and numerous sick days for my mental well-being.  At the same time I have struggled with mismanagement at my job.  One situation was due to higher ups not respecting my confidentiality and they dragged me through the mud, forcing me to get union involvement.  Coworkers have also been getting away with a lot of things to the detriment of my health.  I\'ve been yelled at, I\'ve been subjected to repeated oppressive statements, I\'ve dealt with a coworker assuming more authority than they\'re given and giving everyone and attitude.  They also sexually harassed another coworker.  One person is plainly abusi

# Remove extra fields

In [11]:
loaded = loaded.remove_columns(
    [c for c in loaded.column_names if c not in ["anchor", "positive", "negative", "domain"]]
)

loaded = loaded.map(lambda x: {"domain": x["domain"].replace("learned_hands_", "")})

Map: 100%|██████████| 5603/5603 [00:00<00:00, 21844.05 examples/s]


In [12]:
loaded.save_to_disk("full_triplet_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 5603/5603 [00:00<00:00, 400222.85 examples/s]


# Split into train and test sets

In [13]:
split = loaded.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]
holdout_dataset = split["test"]

split_holdout = holdout_dataset.train_test_split(test_size=0.5, seed=42)

eval_dataset = split_holdout["train"]
test_dataset = split_holdout["test"]

In [14]:
train_dataset.save_to_disk("data/train")
eval_dataset.save_to_disk("data/eval")
test_dataset.save_to_disk("data/test")

Saving the dataset (1/1 shards): 100%|██████████| 281/281 [00:00<00:00, 70238.34 examples/s]


# Quick Analysis

In [15]:
from datasets import load_from_disk

train_dataset = load_from_disk("data/train")
eval_dataset = load_from_disk("data/eval")
test_dataset = load_from_disk("data/test")

In [16]:
print("train:", len(train_dataset))
print("eval :", len(eval_dataset))
print("test :", len(test_dataset))
print("total:", len(train_dataset) + len(eval_dataset) + len(test_dataset))

train: 5042
eval : 280
test : 281
total: 5603
